In [ ]:
from pathlib import Path
from time import perf_counter

import joblib
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import shap
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.neural_network import MLPClassifier
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

ROOT = Path(r"C:\tcc\network-ids-ml-generalization")
DATA_DIR = ROOT / "datasets" / "final" / "ml"
MODEL_DIR = ROOT / "models" / "randomized_search"
EVALUATION_FILE = ROOT / "results" / "model_evaluation" / "evaluation_summary.csv"
OUTPUT_DIR = ROOT / "results" / "shap_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 1
BACKGROUND_SIZE = 200
GLOBAL_SAMPLE_SIZE = 500
MAX_DISPLAY = 15
EXPORT_STATIC_IMAGES = False
LOCAL_CASES = ["true_positive", "true_negative", "false_positive", "false_negative"]

TRAINING_CONFIGS = {
    "cicids2017": ["cicids2017"],
    "unsw_nb15": ["unsw_nb15"],
    "iot23": ["iot23"],
    "cicids2017_unsw_nb15": ["cicids2017", "unsw_nb15"],
    "cicids2017_iot23": ["cicids2017", "iot23"],
    "unsw_nb15_iot23": ["unsw_nb15", "iot23"],
}

EXPLANATION_CONFIG_TEMPLATE = {
    "model": "xgboost",
    "training_source": "cicids2017",
    "test_source": "unsw_nb15",
    "background_size": BACKGROUND_SIZE,
    "global_sample_size": GLOBAL_SAMPLE_SIZE,
    "local_cases": LOCAL_CASES,
}

EXPLANATION_CONFIGS = []

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)


In [ ]:
if not EVALUATION_FILE.exists():
    raise FileNotFoundError(f"Resultados de avaliação não encontrados: {EVALUATION_FILE}")

evaluation_df = pd.read_csv(EVALUATION_FILE)
available_explanations = evaluation_df[[
    "model",
    "training_source",
    "test_source",
    "scenario_type",
    "f1_macro",
    "balanced_accuracy",
    "false_positive",
    "false_negative",
    "model_file",
]].sort_values(
    ["scenario_type", "f1_macro"],
    ascending=[True, False],
).reset_index(drop=True)

display(pd.DataFrame([EXPLANATION_CONFIG_TEMPLATE]))
display(available_explanations)


In [ ]:
reference_model_path = MODEL_DIR / "xgboost" / "cicids2017.joblib"
reference_pipeline = joblib.load(reference_model_path)
reference_preprocessor = reference_pipeline.named_steps["preprocessor"]
numerical_features = list(reference_preprocessor.transformers[0][2])
categorical_features = list(reference_preprocessor.transformers[1][2])
feature_columns = numerical_features + categorical_features
del reference_pipeline, reference_preprocessor

def stratified_sample(dataframe, strata, sample_size, random_state):
    if sample_size is None or sample_size >= len(dataframe):
        return dataframe.copy()

    splitter = StratifiedShuffleSplit(
        n_splits=1,
        train_size=sample_size,
        random_state=random_state,
    )
    sample_indices, _ = next(
        splitter.split(np.zeros(len(dataframe)), strata)
    )
    return dataframe.iloc[np.sort(sample_indices)].copy()


def load_training_background(training_source, sample_size):
    dataset_names = TRAINING_CONFIGS[training_source]
    columns_to_load = feature_columns + ["label_binary", "dataset_name"]
    training_df = pd.concat(
        [
            pd.read_parquet(
                DATA_DIR / f"{dataset_name}_train.parquet",
                columns=columns_to_load,
            )
            for dataset_name in dataset_names
        ],
        ignore_index=True,
    )
    strata = (
        training_df["dataset_name"].astype("string")
        + "__"
        + training_df["label_binary"].astype("string")
    )
    return stratified_sample(
        training_df,
        strata,
        sample_size,
        RANDOM_STATE,
    )


def load_test_data(test_source):
    columns_to_load = feature_columns + ["label_binary"]
    test_df = pd.read_parquet(
        DATA_DIR / f"{test_source}_test.parquet",
        columns=columns_to_load,
    ).reset_index(drop=True)
    test_df.insert(0, "row_id", np.arange(len(test_df)))
    return test_df


def predict_with_scores(pipeline, features):
    predictions = pipeline.predict(features).astype("int8")
    classifier = pipeline.named_steps["classifier"]
    if hasattr(classifier, "predict_proba"):
        class_index = list(classifier.classes_).index(1)
        scores = pipeline.predict_proba(features)[:, class_index]
        score_type = "attack_probability"
    else:
        scores = pipeline.decision_function(features)
        score_type = "decision_margin"
    return predictions, scores, score_type


def add_prediction_metadata(test_df, pipeline):
    result = test_df.copy()
    target = result["label_binary"].eq("ATTACK").astype("int8")
    predictions, scores, score_type = predict_with_scores(
        pipeline,
        result[feature_columns],
    )
    outcome = np.select(
        [
            target.eq(1) & (predictions == 1),
            target.eq(0) & (predictions == 0),
            target.eq(0) & (predictions == 1),
            target.eq(1) & (predictions == 0),
        ],
        [
            "true_positive",
            "true_negative",
            "false_positive",
            "false_negative",
        ],
        default="unknown",
    )
    result["target"] = target
    result["prediction"] = predictions
    result["prediction_score"] = scores
    result["score_type"] = score_type
    result["outcome"] = outcome
    return result


def prepare_explanation_sample(test_df, global_sample_size, local_cases):
    local_frames = []
    selected_row_ids = []

    for case_number, outcome in enumerate(local_cases):
        candidates = test_df.loc[test_df["outcome"].eq(outcome)]
        if candidates.empty:
            continue
        selected = candidates.sample(
            n=1,
            random_state=RANDOM_STATE + case_number,
        ).copy()
        selected["sample_role"] = outcome
        local_frames.append(selected)
        selected_row_ids.extend(selected["row_id"].tolist())

    remaining = test_df.loc[~test_df["row_id"].isin(selected_row_ids)]
    global_sample = stratified_sample(
        remaining,
        remaining["label_binary"],
        global_sample_size,
        RANDOM_STATE,
    )
    global_sample["sample_role"] = "global"

    sample_frames = [global_sample] + local_frames
    return pd.concat(sample_frames, ignore_index=True)


In [ ]:
def transform_features(pipeline, features):
    preprocessor = pipeline.named_steps["preprocessor"]
    transformed = preprocessor.transform(features)
    if hasattr(transformed, "toarray"):
        transformed = transformed.toarray()
    feature_names = preprocessor.get_feature_names_out().tolist()
    return pd.DataFrame(
        transformed,
        columns=feature_names,
        index=features.index,
    )


def select_attack_output(explanation, classifier):
    values = np.asarray(explanation.values)
    base_values = np.asarray(explanation.base_values)

    if values.ndim == 3:
        class_index = list(classifier.classes_).index(1)
        values = values[:, :, class_index]
        if base_values.ndim == 2:
            base_values = base_values[:, class_index]
        elif base_values.ndim == 1 and len(base_values) == len(classifier.classes_):
            base_values = np.repeat(base_values[class_index], len(values))

    if base_values.ndim == 0:
        base_values = np.repeat(float(base_values), len(values))

    return shap.Explanation(
        values=values,
        base_values=base_values,
        data=np.asarray(explanation.data),
        feature_names=list(explanation.feature_names),
        output_names="ATTACK",
    )


def explain_transformed_features(pipeline, background, sample):
    classifier = pipeline.named_steps["classifier"]
    feature_names = sample.columns.tolist()

    if isinstance(
        classifier,
        (DecisionTreeClassifier, RandomForestClassifier, XGBClassifier),
    ):
        explainer = shap.TreeExplainer(
            classifier,
            data=background,
            model_output="probability",
            feature_perturbation="interventional",
            feature_names=feature_names,
        )
        explanation = explainer(sample, check_additivity=False)
        output_scale = "attack_probability"
    elif isinstance(classifier, LinearSVC):
        masker = shap.maskers.Independent(background)
        explainer = shap.LinearExplainer(classifier, masker)
        explanation = explainer(sample)
        output_scale = "decision_margin"
    else:
        masker = shap.maskers.Independent(background)
        if hasattr(classifier, "predict_proba"):
            class_index = list(classifier.classes_).index(1)

            def prediction_function(values):
                return classifier.predict_proba(np.asarray(values))[:, class_index]

            output_scale = "attack_probability"
        else:
            def prediction_function(values):
                return classifier.decision_function(np.asarray(values))

            output_scale = "decision_margin"

        explainer = shap.Explainer(
            prediction_function,
            masker,
            algorithm="permutation",
            feature_names=feature_names,
            output_names=["ATTACK"],
            seed=RANDOM_STATE,
        )
        explanation = explainer(
            sample,
            max_evals=2 * len(feature_names) + 1,
            silent=True,
        )

    return select_attack_output(explanation, classifier), output_scale


def original_feature_name(transformed_feature_name):
    feature_suffix = transformed_feature_name.split("__", 1)[-1]
    for feature_name in sorted(feature_columns, key=len, reverse=True):
        if feature_suffix == feature_name:
            return feature_name
        if feature_suffix.startswith(f"{feature_name}_"):
            return feature_name
    return feature_suffix


def aggregate_original_features(explanation, original_features):
    feature_mapping = pd.DataFrame({
        "transformed_feature": explanation.feature_names,
        "original_feature": [
            original_feature_name(feature_name)
            for feature_name in explanation.feature_names
        ],
    })
    ordered_original_features = list(dict.fromkeys(feature_mapping["original_feature"]))
    aggregated_values = np.column_stack([
        explanation.values[:, feature_mapping["original_feature"].eq(feature_name)].sum(axis=1)
        for feature_name in ordered_original_features
    ])
    display_data = original_features[ordered_original_features].copy()
    for feature_name in display_data:
        if not pd.api.types.is_numeric_dtype(display_data[feature_name]):
            display_data[feature_name] = pd.factorize(display_data[feature_name])[0]

    aggregated_explanation = shap.Explanation(
        values=aggregated_values,
        base_values=explanation.base_values,
        data=display_data.to_numpy(),
        feature_names=ordered_original_features,
        output_names="ATTACK",
    )
    return aggregated_explanation, feature_mapping


In [ ]:
def save_plotly_figure(figure, output_directory, filename, width, height):
    figure.write_html(
        output_directory / f"{filename}.html",
        include_plotlyjs="cdn",
    )
    figure.write_json(output_directory / f"{filename}.json")
    if EXPORT_STATIC_IMAGES:
        figure.write_image(
            output_directory / f"{filename}.png",
            width=width,
            height=height,
            scale=2,
        )
        figure.write_image(
            output_directory / f"{filename}.svg",
            width=width,
            height=height,
        )
    display(figure)


def save_explanation_tables(
    output_directory,
    sample_metadata,
    explanation,
    feature_mapping,
    output_scale,
):
    metadata_columns = [
        "row_id",
        "label_binary",
        "target",
        "prediction",
        "prediction_score",
        "score_type",
        "outcome",
        "sample_role",
    ]
    sample_metadata[metadata_columns].to_csv(
        output_directory / "explained_samples.csv",
        index=False,
    )
    feature_mapping.to_csv(
        output_directory / "feature_mapping.csv",
        index=False,
    )

    shap_frame = pd.DataFrame(
        explanation.values,
        columns=explanation.feature_names,
    )
    shap_frame.insert(0, "row_id", sample_metadata["row_id"].to_numpy())
    shap_long = shap_frame.melt(
        id_vars="row_id",
        var_name="feature",
        value_name="shap_value",
    )
    shap_long["absolute_shap_value"] = shap_long["shap_value"].abs()
    shap_long["output_scale"] = output_scale
    shap_long.to_csv(output_directory / "shap_values.csv", index=False)

    global_positions = np.flatnonzero(sample_metadata["sample_role"].eq("global"))
    global_values = explanation.values[global_positions]
    global_importance = pd.DataFrame({
        "feature": explanation.feature_names,
        "mean_absolute_shap": np.abs(global_values).mean(axis=0),
        "mean_signed_shap": global_values.mean(axis=0),
        "output_scale": output_scale,
    }).sort_values("mean_absolute_shap", ascending=False)
    global_importance.to_csv(
        output_directory / "global_importance.csv",
        index=False,
    )
    return global_positions, global_importance


def normalized_feature_values(values):
    numeric_values = pd.to_numeric(pd.Series(values), errors="coerce")
    if numeric_values.isna().all():
        numeric_values = pd.Series(pd.factorize(values)[0], dtype=float)
    else:
        numeric_values = numeric_values.fillna(numeric_values.median())
    minimum = numeric_values.min()
    maximum = numeric_values.max()
    if minimum == maximum:
        return np.repeat(0.5, len(numeric_values))
    return ((numeric_values - minimum) / (maximum - minimum)).to_numpy()


def build_global_bar_figure(global_importance):
    plot_data = (
        global_importance.head(MAX_DISPLAY)
        .sort_values("mean_absolute_shap")
    )
    figure = px.bar(
        plot_data,
        x="mean_absolute_shap",
        y="feature",
        orientation="h",
        labels={
            "mean_absolute_shap": "Média do valor SHAP absoluto",
            "feature": "Característica",
        },
        title="Importância global das características",
    )
    figure.update_traces(
        marker_color="#4C78A8",
        hovertemplate=(
            "Característica: %{y}<br>"
            "Média |SHAP|: %{x:.6f}<extra></extra>"
        ),
    )
    figure.update_layout(template="plotly_white", showlegend=False)
    return figure


def build_global_beeswarm_figure(explanation, global_positions, global_importance):
    top_features = global_importance.head(MAX_DISPLAY)["feature"].tolist()
    random_generator = np.random.default_rng(RANDOM_STATE)
    beeswarm_frames = []

    for feature_order, feature_name in enumerate(top_features):
        feature_index = explanation.feature_names.index(feature_name)
        feature_values = explanation.data[global_positions, feature_index]
        shap_values = explanation.values[global_positions, feature_index]
        beeswarm_frames.append(pd.DataFrame({
            "feature": feature_name,
            "feature_order": feature_order,
            "feature_position": (
                feature_order
                + random_generator.uniform(-0.28, 0.28, len(global_positions))
            ),
            "feature_value": feature_values,
            "normalized_feature_value": normalized_feature_values(feature_values),
            "shap_value": shap_values,
        }))

    plot_data = pd.concat(beeswarm_frames, ignore_index=True)
    custom_data = plot_data[["feature", "feature_value"]].to_numpy()
    figure = go.Figure(go.Scatter(
        x=plot_data["shap_value"],
        y=plot_data["feature_position"],
        mode="markers",
        customdata=custom_data,
        marker={
            "color": plot_data["normalized_feature_value"],
            "colorscale": "RdBu_r",
            "cmin": 0,
            "cmax": 1,
            "size": 7,
            "opacity": 0.75,
            "colorbar": {
                "title": "Valor da característica",
                "tickmode": "array",
                "tickvals": [0, 1],
                "ticktext": ["Baixo", "Alto"],
            },
        },
        hovertemplate=(
            "Característica: %{customdata[0]}<br>"
            "Valor: %{customdata[1]}<br>"
            "SHAP: %{x:.6f}<extra></extra>"
        ),
    ))
    figure.add_vline(x=0, line_width=1, line_color="#666666")
    figure.update_yaxes(
        tickmode="array",
        tickvals=list(range(len(top_features))),
        ticktext=top_features,
        range=[len(top_features) - 0.5, -0.5],
        title_text="Característica",
    )
    figure.update_xaxes(title_text="Valor SHAP")
    figure.update_layout(
        template="plotly_white",
        title="Distribuição global dos valores SHAP",
        showlegend=False,
    )
    return figure


def build_local_waterfall_figure(explanation, position, sample_metadata):
    contribution_values = explanation.values[position]
    feature_order = np.argsort(np.abs(contribution_values))[::-1]
    selected_indices = feature_order[: max(1, MAX_DISPLAY - 1)]
    remaining_indices = feature_order[max(1, MAX_DISPLAY - 1):]
    labels = [explanation.feature_names[index] for index in selected_indices]
    contributions = contribution_values[selected_indices].tolist()
    if len(remaining_indices):
        labels.append("Demais características")
        contributions.append(float(contribution_values[remaining_indices].sum()))

    base_values = np.asarray(explanation.base_values)
    base_value = float(base_values[position] if base_values.ndim else base_values)
    sample_role = sample_metadata.iloc[position]["sample_role"]
    row_id = sample_metadata.iloc[position]["row_id"]
    figure = go.Figure(go.Waterfall(
        orientation="h",
        measure=["absolute"] + ["relative"] * len(contributions) + ["total"],
        y=["Valor base"] + labels + ["Saída do modelo"],
        x=[base_value] + contributions + [0],
        connector={"line": {"color": "#777777"}},
        increasing={"marker": {"color": "#D62728"}},
        decreasing={"marker": {"color": "#1F77B4"}},
        totals={"marker": {"color": "#666666"}},
        hovertemplate="%{y}<br>Contribuição: %{x:.6f}<extra></extra>",
    ))
    figure.update_layout(
        template="plotly_white",
        title=f"Explicação local: {sample_role} · linha {row_id}",
        xaxis_title="Saída explicada pelo SHAP",
        showlegend=False,
    )
    return figure


def save_explanation_plots(
    output_directory,
    sample_metadata,
    explanation,
    global_positions,
    global_importance,
):
    bar_figure = build_global_bar_figure(global_importance)
    save_plotly_figure(bar_figure, output_directory, "global_bar", 1100, 650)

    beeswarm_figure = build_global_beeswarm_figure(
        explanation,
        global_positions,
        global_importance,
    )
    save_plotly_figure(
        beeswarm_figure,
        output_directory,
        "global_beeswarm",
        1200,
        750,
    )

    local_positions = np.flatnonzero(sample_metadata["sample_role"].ne("global"))
    for position in local_positions:
        sample_role = sample_metadata.iloc[position]["sample_role"]
        row_id = sample_metadata.iloc[position]["row_id"]
        waterfall_figure = build_local_waterfall_figure(
            explanation,
            position,
            sample_metadata,
        )
        save_plotly_figure(
            waterfall_figure,
            output_directory,
            f"local_{sample_role}_row_{row_id}",
            1200,
            750,
        )


In [ ]:
def validate_explanation_config(config):
    required_keys = {"model", "training_source", "test_source"}
    missing_keys = required_keys.difference(config)
    if missing_keys:
        raise ValueError(f"Configuração incompleta: {sorted(missing_keys)}")
    if config["training_source"] not in TRAINING_CONFIGS:
        raise ValueError(f"Origem de treinamento inválida: {config['training_source']}")
    available = evaluation_df.loc[
        evaluation_df["model"].eq(config["model"])
        & evaluation_df["training_source"].eq(config["training_source"])
        & evaluation_df["test_source"].eq(config["test_source"])
    ]
    if available.empty:
        raise ValueError(f"Combinação não encontrada na avaliação: {config}")


def run_shap_analysis(config):
    validate_explanation_config(config)
    model_name = config["model"]
    training_source = config["training_source"]
    test_source = config["test_source"]
    background_size = config.get("background_size", BACKGROUND_SIZE)
    global_sample_size = config.get("global_sample_size", GLOBAL_SAMPLE_SIZE)
    local_cases = config.get("local_cases", LOCAL_CASES)

    model_path = MODEL_DIR / model_name / f"{training_source}.joblib"
    if not model_path.exists():
        raise FileNotFoundError(f"Modelo não encontrado: {model_path}")

    output_directory = OUTPUT_DIR / model_name / f"{training_source}__{test_source}"
    output_directory.mkdir(parents=True, exist_ok=True)

    analysis_start = perf_counter()
    pipeline = joblib.load(model_path)
    background_df = load_training_background(training_source, background_size)
    test_df = add_prediction_metadata(load_test_data(test_source), pipeline)
    explanation_sample = prepare_explanation_sample(
        test_df,
        global_sample_size,
        local_cases,
    )

    transformed_background = transform_features(
        pipeline,
        background_df[feature_columns],
    )
    transformed_sample = transform_features(
        pipeline,
        explanation_sample[feature_columns],
    )
    transformed_explanation, output_scale = explain_transformed_features(
        pipeline,
        transformed_background,
        transformed_sample,
    )
    explanation, feature_mapping = aggregate_original_features(
        transformed_explanation,
        explanation_sample[feature_columns],
    )

    global_positions, global_importance = save_explanation_tables(
        output_directory,
        explanation_sample,
        explanation,
        feature_mapping,
        output_scale,
    )
    save_explanation_plots(
        output_directory,
        explanation_sample,
        explanation,
        global_positions,
        global_importance,
    )

    elapsed_seconds = perf_counter() - analysis_start
    result = {
        "model": model_name,
        "training_source": training_source,
        "test_source": test_source,
        "background_rows": len(background_df),
        "global_rows": len(global_positions),
        "local_rows": int(explanation_sample["sample_role"].ne("global").sum()),
        "output_scale": output_scale,
        "top_feature": global_importance.iloc[0]["feature"],
        "elapsed_seconds": elapsed_seconds,
        "output_directory": str(output_directory.relative_to(ROOT)),
    }
    pd.DataFrame([result]).to_csv(
        output_directory / "analysis_summary.csv",
        index=False,
    )
    return result


In [ ]:
if not EXPLANATION_CONFIGS:
    print("Nenhum caso selecionado. Preencha EXPLANATION_CONFIGS para executar a análise SHAP.")
else:
    analysis_results = []
    for config_number, config in enumerate(EXPLANATION_CONFIGS, start=1):
        result = run_shap_analysis(config)
        analysis_results.append(result)
        print(
            f"{config_number:02d}/{len(EXPLANATION_CONFIGS):02d} - "
            f"{result['model']} - {result['training_source']} - "
            f"{result['test_source']} - {result['elapsed_seconds'] / 60:.2f} min"
        )

    analysis_results_df = pd.DataFrame(analysis_results)
    analysis_results_df.to_csv(OUTPUT_DIR / "analysis_summary.csv", index=False)
    display(analysis_results_df)
